# Brent · `application_type` (web: "Application type")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `application_type` values

A `terms` aggregation on `application_type.raw` (exact values, case-sensitive) plus a
`missing` aggregation for records with no value. The `shown_as` column uses `repr()` so
hidden characters (tabs, spaces) are visible.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": brent_filter(),
    "aggs": {
        "application_type": {"terms": {"field": "application_type.raw", "size": 200}},
        "no_application_type": {"missing": {"field": "application_type.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["application_type"]["buckets"]
type_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_application_type"]["doc_count"])],
    columns=["application_type", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Brent applications:", total, "| sum of counts:", type_counts["count"].sum())

Brent applications: 36952 | sum of counts: 36952


In [3]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "application_type" (extracted 2026-11-09).
SCHEMA_APPLICATION_TYPE = ["Householder", "Prior Approval", "All Other"]

type_counts["shown_as"] = type_counts["application_type"].map(repr)
type_counts["in_schema"] = type_counts["application_type"].isin(SCHEMA_APPLICATION_TYPE)
type_counts["share_%"] = (100 * type_counts["count"] / total).round(2)
type_counts

,application_type,count,shown_as,in_schema,share_%
0,All Other,23336,'All Other',True,63.15
1,Householder,9524,'Householder',True,25.77
2,Prior Approval,4092,'Prior Approval',True,11.07
3,NaN,0,nan,False,0.00


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| `application_type` | Count | Share | In schema |
|---|---|---|---|
| All Other | 23,336 | 63.15% | yes |
| Householder | 9,524 | 25.77% | yes |
| Prior Approval | 4,092 | 11.07% | yes |
| null | 0 | 0% | — |

- **Clean:** every record has a value, all 3 are the schema values, and there are no case
  or whitespace variants.
- **Only 3 broad groups:** `All Other` covers 63% of applications. The schema also has
  `application_type_full` (e.g. "Householder planning permission"), which is likely where
  the detailed type is.

## Step 2 · Decisions for `Prior Approval`

"Prior Approval" is a kind of application (asking whether the council's prior approval
is needed for permitted development), not an outcome. Count the `decision` values for
these applications to check that they are not all approved.

In [4]:
body = {
    "size": 0,
    "query": {"bool": {"filter": [brent_filter(), {"term": {"application_type.raw": "Prior Approval"}}]}},
    "aggs": {
        "decision": {"terms": {"field": "decision.raw", "size": 50}},
        "no_decision": {"missing": {"field": "decision.raw"}},
    },
}
aggs = search(body)["aggregations"]
prior_approval = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in aggs["decision"]["buckets"]] + [(None, aggs["no_decision"]["doc_count"])],
    columns=["decision", "count"],
)
prior_approval["shown_as"] = prior_approval["decision"].map(repr)
prior_approval["share_%"] = (100 * prior_approval["count"] / prior_approval["count"].sum()).round(2)
print("Prior Approval applications:", prior_approval["count"].sum())
prior_approval

Prior Approval applications: 4092


,decision,count,shown_as,share_%
0,Approved,2666,'Approved',65.15
1,Refused,1285,'Refused',31.40
2,Withdrawn,95,'Withdrawn',2.32
3,No Objection to Proposal (OBS only),4,'No Objection to Proposal (OBS only)',0.10
4,\tApproved,2,'\tApproved',0.05
5,Objection Raised to Proposal (OBS only),2,'Objection Raised to Proposal (OBS only)',0.05
6,NaN,38,nan,0.93


### Findings · Step 2

Live API, 2026-11-09, 4,092 `Prior Approval` applications:

| `decision` | Count | Share |
|---|---|---|
| Approved (incl. 2 `'\tApproved'`) | 2,668 | 65.2% |
| Refused | 1,285 | 31.4% |
| Withdrawn | 95 | 2.3% |
| No Objection / Objection Raised (OBS only) | 6 | 0.15% |
| null | 38 | 0.9% |

- **Not all approved:** almost a third were refused. "Prior Approval" is the kind of
  application, not its outcome; the outcome is in `decision`.

## Step 3 · All `application_type_full` values

`application_type_full` is not a web column, but it holds the detailed type behind
`application_type`. Count its values per `application_type` (a `terms` aggregation on
`application_type.raw` with a `terms` aggregation on `application_type_full.raw` inside),
plus records with no value.

In [5]:
body = {
    "size": 0,
    "query": brent_filter(),
    "aggs": {
        "type": {
            "terms": {"field": "application_type.raw", "size": 10},
            "aggs": {
                "full": {"terms": {"field": "application_type_full.raw", "size": 500}},
                "no_full": {"missing": {"field": "application_type_full.raw"}},
            },
        },
    },
}
aggs = search(body)["aggregations"]

rows = []
for t in aggs["type"]["buckets"]:
    rows += [(t["key"], b["key"], b["doc_count"]) for b in t["full"]["buckets"]]
    rows.append((t["key"], None, t["no_full"]["doc_count"]))
full_counts = pd.DataFrame(rows, columns=["application_type", "application_type_full", "count"])
full_counts = full_counts[full_counts["count"] > 0].reset_index(drop=True)
print("sum of counts:", full_counts["count"].sum(), "| distinct values:", full_counts["application_type_full"].nunique())

sum of counts: 36952 | distinct values: 39


In [6]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "application_type_full" (extracted 2026-11-09).
SCHEMA_APPLICATION_TYPE_FULL = [
    'Householder planning permission',
    'Householder planning & demolition in a conservation area',
    'Householder planning & listed building consent',
    'Full planning permission',
    'Outline planning permission: Some matters reserved',
    'Outline planning permission: All matters reserved',
    'Full planning & demolition in a conservation area',
    'Full planning & listed building consent',
    'Full planning & display of advertisements',
    'Lawful development: Existing use',
    'Lawful development: Proposed use',
    'Prior Approval: Building for agricultural/forestry use',
    'Prior Approval: Private road for agricultural/forestry use',
    'Prior Approval: Excavation/Deposit waste for agriculture',
    'Prior Approval: Tank/Cage/Structure for use in fish farming',
    'Prior Approval: Development for electronic communications network',
    'Prior Approval: Demolition of building',
    'Approval of reserved matters',
    'Prior Approval: Larger Home Extension',
    'Prior Approval: Change of use - retail to restaurant/café',
    'Prior Approval: Change of use - retail/takeaway to dwellinghouses',
    'Prior Approval: Change of use - retail to assembly/leisure',
    'Prior Approval: Change of use - agriculture to dwellinghouses',
    'Prior Approval: Change of use - agriculture to flexible commercial use',
    'Prior Approval: Change of use - agriculture to school/nursery',
    'Prior Approval: Change of use - business/hotels/etc to school/nursery',
    'Prior Approval: Change of use - offices to dwellinghouses',
    'Prior Approval: Change of use - light industrial to dwellinghouses',
    'Prior Approval: Change of use - amusements/casinos to dwellinghouses',
    'Prior Approval: Collection facility for a shop',
    'Prior Approval: Roof mounted solar PV on non-domestic building',
    'Prior Approval: Temporary school on previously vacant commercial land',
    'Prior Approval: Temporary use for commercial film-making',
    'Prior Approval: Change of use - retail/takeaway to offices',
]

full_counts["shown_as"] = full_counts["application_type_full"].map(repr)
full_counts["in_schema"] = full_counts["application_type_full"].isin(SCHEMA_APPLICATION_TYPE_FULL)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 100)
full_counts

,application_type,application_type_full,count,shown_as,in_schema
0,All Other,Full planning permission,8383,'Full planning permission',True
1,All Other,Lawful development: Proposed use,2758,'Lawful development: Proposed use',True
2,All Other,Approval of details reserved by a condition (discharge),1860,'Approval of details reserved by a condition (discharge)',False
3,All Other,Lawful development: Existing use,722,'Lawful development: Existing use',True
4,All Other,Non-Material Amendment,553,'Non-Material Amendment',False
5,All Other,Removal/Variation of a condition,326,'Removal/Variation of a condition',False
6,All Other,Consent to display an advertisement,279,'Consent to display an advertisement',False
7,All Other,Full planning & display of advertisements,246,'Full planning & display of advertisements',True
8,All Other,Outline planning permission: Some matters reserved,172,'Outline planning permission: Some matters reserved',True
9,All Other,Householder planning permission,114,'Householder planning permission',True


### Findings · Step 3

Live API, 2026-11-09, 36,952 Brent applications:

| | Records | Distinct values |
|---|---|---|
| null | 13,223 (35.8%) | — |
| value in the schema list | 20,533 | 19 |
| value not in the schema list | 3,196 | 20 |

null by `application_type`: All Other 7,641, Householder 3,546, Prior Approval 2,036.

- **A third have no detailed type.** For those, only the 3 broad groups are known.
- **The schema list is incomplete, not the data.** The 20 values not in the list are real
  application types, e.g. `Approval of details reserved by a condition (discharge)` (1,861),
  `Non-Material Amendment` (554), `Removal/Variation of a condition` (347),
  `Consent to display an advertisement` (279), and newer Prior Approval types
  (Use Class E, new dwellinghouses on blocks of flats). No case or whitespace variants.
- **15 schema values are never used**, all Prior Approval types (mostly agricultural).
- **The two fields don't always agree:** 541 `Householder` records have
  `Full planning permission`, 114 `All Other` have `Householder planning permission`,
  43 `Householder` have `Prior Approval: Larger Home Extension`, and 1 `All Other` has
  `Prior Approval: Demolition of building`.

## Step 4 · Do the values match the schema type?

The schema gives `application_type` the type `string`. Earlier counts used an Elasticsearch
aggregation (or `value_counts`), which can't show whether a value came as a JSON string or a
JSON number. Count the Python type of every value: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [7]:
values, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "application_type"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    values += [h["_source"] for h in page]
    after = page[-1]["sort"]

typed = pd.DataFrame(values).reindex(columns=["id", "application_type"])
types = typed["application_type"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Brent applications:", len(typed))
display(types.value_counts().to_frame("count"))

not_str = typed[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["application_type"].value_counts().head(30).to_frame("count")

Brent applications: 36952


,count
application_type,
str,36952


values that are not strings: 0


,count
application_type,


### Findings · Step 4

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 36,952 |
| null | 0 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says. No type mismatch.